# 14. Missing Values: NaN and inf

Real data has gaps. NumPy doesn't have a dedicated "missing" value for numbers, so the convention is to use `np.nan` (not a number). This notebook covers how NaN behaves and how to work with it.

In [1]:
import numpy as np

data = np.array([2.0, np.nan, 5.0, 1.0, np.nan, 7.0])
data

array([ 2., nan,  5.,  1., nan,  7.])

## NaN spreads

Any arithmetic involving NaN gives NaN, so one missing value ruins a normal sum or mean:

In [2]:
print(data.sum(), data.mean(), data.max())

nan nan nan


## NaN isn't equal to anything, including itself

So `data == np.nan` never finds anything. Use `np.isnan`:

In [3]:
print(np.nan == np.nan)
print(data == np.nan)
print(np.isnan(data))

False
[False False False False False False]
[False  True False False  True False]


In [4]:
print('missing:', np.isnan(data).sum())
print('positions:', np.flatnonzero(np.isnan(data)))

missing: 2
positions: [1 4]


NaN only exists for floats. An integer array can't hold it, which is why integer data with gaps usually ends up as floats:

In [5]:
try:
    np.array([1, 2, 3])[0] = np.nan
except ValueError as e:
    print('ValueError:', e)

ValueError: cannot convert float NaN to integer


## NaN-aware functions

Most reductions have a `nan*` version that skips missing values:

In [6]:
print(np.nansum(data))
print(np.nanmean(data))
print(np.nanmax(data), np.nanmin(data))
print(np.nanstd(data))
print(np.nanmedian(data))
print(np.nanpercentile(data, 50))
print(np.nanargmax(data))

15.0
3.75
7.0 1.0
2.384848003542364
3.5
3.5
5


They work along axes too:

In [7]:
table = np.array([[1.0, np.nan, 3.0],
                  [4.0, 5.0, 6.0],
                  [np.nan, 8.0, 9.0]])
print(np.nanmean(table, axis=0))

[2.5 6.5 6. ]


## Dropping missing values

In [8]:
print(data[~np.isnan(data)])

[2. 5. 1. 7.]


In [9]:
# drop rows that contain any NaN
rows_ok = ~np.isnan(table).any(axis=1)
table[rows_ok]

array([[4., 5., 6.]])

## Filling missing values

In [10]:
print(np.nan_to_num(data, nan=0.0))
print(np.where(np.isnan(data), np.nanmean(data), data))

[2. 0. 5. 1. 0. 7.]
[2.   3.75 5.   1.   3.75 7.  ]


Filling each column with its own mean, using broadcasting:

In [11]:
col_means = np.nanmean(table, axis=0)
filled = np.where(np.isnan(table), col_means, table)
filled

array([[1. , 6.5, 3. ],
       [4. , 5. , 6. ],
       [2.5, 8. , 9. ]])

Linear interpolation for gaps in a 1-D series:

In [12]:
series = np.array([1.0, np.nan, np.nan, 4.0, 5.0, np.nan, 7.0])
idx = np.arange(len(series))
known = ~np.isnan(series)
np.interp(idx, idx[known], series[known])

array([1., 2., 3., 4., 5., 6., 7.])

## Infinity

Dividing by zero gives `inf` (or `nan` for 0/0). NumPy warns instead of raising. `np.isinf` and `np.isfinite` let you check for them:

In [13]:
with np.errstate(divide='ignore', invalid='ignore'):
    x = np.array([1.0, 0.0, -1.0]) / 0.0
print(x)
print(np.isinf(x), np.isnan(x), np.isfinite(x))

[ inf  nan -inf]
[ True False  True] [False  True False] [False False False]


`np.errstate` controls what happens on these floating point problems. You can silence the warnings (as above) or turn them into errors, which is great for tracking down where a NaN first appears:

In [14]:
try:
    with np.errstate(all='raise'):
        np.log(np.array([0.0]))
except FloatingPointError as e:
    print('FloatingPointError:', e)

FloatingPointError: divide by zero encountered in log


## Masked arrays

`numpy.ma` is the other approach: keep the data as is and attach a mask saying which values to ignore. It works with any dtype, including integers, and most functions respect the mask automatically.

In [15]:
readings = np.array([12, -999, 15, 14, -999, 13])    # -999 = sensor error
m = np.ma.masked_equal(readings, -999)
print(m)
print(m.mean())
print(m.filled(0))

[12 -- 15 14 -- 13]
13.5
[12  0 15 14  0 13]


Honestly, for most tabular data with lots of missing values, pandas is more comfortable. But for numerical arrays, NaN plus the `nan*` functions covers almost everything.